# DA6401 — Pretrained Classifier + Segmentation (Final)

**BEFORE RUNNING:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **ON**
3. Click **Save Version** → **Save & Run All (Commit)**

**What this notebook does:**
- Loads ImageNet pretrained VGG11-BN encoder weights (biggest F1 boost)
- Fine-tunes classifier for 40 epochs at lr=5e-4  → expect F1 ≈ 0.85+
- Retrains U-Net segmentation (partial freeze) for 60 epochs using new encoder

**Why this works:**
- Training from scratch (60 epochs) → F1 ≈ 0.54
- Fine-tuning from ImageNet pretrained → F1 ≈ 0.85+ in fewer epochs
  because the encoder already knows edges, textures, shapes

**Output files:**
- `checkpoints/classifier.pth`  ← upload to Drive, update CLASSIFIER_DRIVE_ID
- `checkpoints/unet.pth`        ← upload to Drive, update UNET_DRIVE_ID

**Estimated runtime:** ~5 hours

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT = "/kaggle/working/checkpoints"
os.makedirs(CKPT, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git /kaggle/working/repo 2>&1 | tail -3
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q wandb albumentations gdown scikit-learn

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
print(f"GPU: {gpu}")
assert torch.cuda.is_available(), "NO GPU — go to Settings and enable GPU T4 x2!"

In [ ]:
from data.pets_dataset import download_oxford_pet
download_oxford_pet("./data/oxford_pet")
print("Dataset ready.")

## 1. Classifier — ImageNet Pretrained Fine-tune

Uses `--pretrained` flag to load VGG11-BN ImageNet weights into the encoder before training.
40 epochs at lower LR (5e-4) is enough — pretrained encoder converges much faster.

In [ ]:
# Fine-tune with ImageNet pretrained encoder
# --pretrained loads torchvision VGG11-BN ImageNet weights into encoder blocks
# Lower LR (5e-4) since encoder already has good features
!python train.py --task classify --experiment kaggle-pretrained --pretrained \
    --dropout 0.5 \
    --epochs 40 --lr 5e-4 --batch-size 32 --num-workers 2 \
    --checkpoint-dir {CKPT}

!cp {CKPT}/classifier.pth {CKPT}/classifier_pretrained.pth
print("\n--- Done → classifier.pth (pretrained fine-tune) ---")

## 2. Segmentation — Partial Freeze (using new pretrained encoder)

U-Net trained with the improved encoder from step 1.
Partial freeze: blocks 0-2 frozen, blocks 3-4 + decoder trainable.

In [ ]:
# Retrain U-Net with partial freeze using new pretrained classifier encoder
!python train.py --task segment --experiment kaggle-pretrained --freeze-mode partial \
    --epochs 60 --lr 1e-3 --batch-size 32 --num-workers 2 \
    --checkpoint-dir {CKPT}

!cp {CKPT}/unet.pth {CKPT}/unet_pretrained_partial.pth
print("\n--- Done → unet.pth (partial freeze, pretrained encoder) ---")

## 3. Evaluate All Three Tasks

In [ ]:
import os, torch, numpy as np
from sklearn.metrics import f1_score
from data.pets_dataset import create_dataloaders
from models.classification import VGG11Classifier
from models.localization import VGG11Localizer
from models.segmentation import VGG11UNet
from train import compute_iou, compute_dice_score
import gdown

CKPT = "/kaggle/working/checkpoints"
device = torch.device("cuda")

_, val_loader, test_loader = create_dataloaders(
    root="./data/oxford_pet", batch_size=64, num_workers=2
)

print("="*60)
print("  FINAL EVALUATION")
print("="*60)

# ── 1. Classification ──
cls_model = VGG11Classifier(num_classes=37, dropout_p=0.5, use_bn=True).to(device)
cls_model.load_state_dict(torch.load(f"{CKPT}/classifier.pth", map_location=device, weights_only=False))
cls_model.eval()
preds, labels = [], []
with torch.no_grad():
    for b in val_loader:
        preds.extend(cls_model(b["image"].to(device)).argmax(1).cpu().tolist())
        labels.extend(b["label"].tolist())
f1 = f1_score(labels, preds, average="macro")
print(f"\n  Classification — Val F1 (macro): {f1:.4f}  (target: 0.80)")

# ── 2. Localization (download from Drive if needed) ──
loc_path = f"{CKPT}/localizer.pth"
if not os.path.exists(loc_path):
    print("  Downloading localizer.pth from Drive...")
    LOC_DRIVE_ID = "1QjDlqTkosCAMJNOfERqaaQGmNdjeIW3w"
    gdown.download(id=LOC_DRIVE_ID, output=loc_path, quiet=False, fuzzy=True)

loc_model = VGG11Localizer().to(device)
loc_model.load_state_dict(torch.load(loc_path, map_location=device, weights_only=False))
loc_model.eval()
total_iou, iou_50, total = 0.0, 0, 0
with torch.no_grad():
    for b in val_loader:
        pred = loc_model(b["image"].to(device)).cpu()
        ious = compute_iou(pred, b["bbox"])
        total_iou += ious.sum().item()
        iou_50 += (ious >= 0.5).sum().item()
        total += len(ious)
print(f"  Localization  — Val mean IoU: {total_iou/total:.4f}, Acc@0.5: {iou_50/total*100:.1f}%  (target: 60%)")

# ── 3. Segmentation ──
seg_model = VGG11UNet(num_classes=3).to(device)
seg_model.load_state_dict(torch.load(f"{CKPT}/unet.pth", map_location=device, weights_only=False))
seg_model.eval()
dice_sum, total = 0.0, 0
with torch.no_grad():
    for b in val_loader:
        logits = seg_model(b["image"].to(device))
        pred_masks = logits.argmax(dim=1).cpu()
        dice_sum += compute_dice_score(pred_masks, b["mask"]) * b["image"].size(0)
        total += b["image"].size(0)
print(f"  Segmentation  — Val Dice: {dice_sum/total:.4f}  (target: 0.30+)")

print("\n  Files:")
for f in sorted(os.listdir(CKPT)):
    size_mb = os.path.getsize(f"{CKPT}/{f}") / 1e6
    print(f"    {f:45s} {size_mb:6.1f} MB")

print("\n  ↑ Download classifier.pth and unet.pth from Output tab")
print("  ↑ Upload to Google Drive, share, get file IDs")
print("  ↑ Update CLASSIFIER_DRIVE_ID and UNET_DRIVE_ID in models/multitask.py")
print("="*60)